# Pipeline ML completo — exploração interativa

Este notebook percorre as **4 etapas do pipeline de Machine Learning** que está nos scripts `.py` da pasta, mas em formato exploratório: com markdown explicando o que está acontecendo, gráficos e insights, e uns "mexa você mesmo" no final de cada parte.

Use este notebook para **aprender o pipeline** e usar os scripts `.py` para **rodar em produção**.

---

## Pré-requisito

Você precisa do `olist.parquet` baixado. Se ainda não estiver, rode o notebook `07-bloco-3-olist-analise.ipynb` ou execute a célula abaixo.

In [ ]:
import urllib.request, pathlib

URL = "https://github.com/fenakamuta/poliusppro-data-engineering/releases/download/aula1-olist-2026-v1/olist.parquet"
PATH = pathlib.Path("../data/olist.parquet")
PATH.parent.mkdir(exist_ok=True)
if not PATH.exists():
    print("Baixando olist.parquet...")
    urllib.request.urlretrieve(URL, PATH)
print(f"OK: {PATH} ({PATH.stat().st_size/1e6:.1f} MB)")

---

## A arquitetura que vamos explorar

```
olist.parquet
     ↓
[ Etapa 1 — Feature Engineering com DuckDB SQL ]
     ↓
features.parquet
     ↓
[ Etapa 2 — Treino com sklearn ]
     ↓
model.joblib
     ↓
[ Etapa 3 — Predição em batch ]
     ↓
predictions.parquet
     ↓
[ Etapa 4 — Avaliação via DuckDB SQL ]
```

Cada etapa tem um equivalente em script `.py` na mesma pasta. Vou referenciar quando for o caso.

---

## Etapa 1 — Feature Engineering com DuckDB SQL

> Equivalente: [`01_features.py`](./01_features.py)

O engenheiro de dados prepara um conjunto pronto para treinar. Isso é feito em **SQL puro** — DuckDB cuida da transformação dos dados brutos do Olist em features numéricas e categóricas.

**O que vamos fazer:**
1. Selecionar colunas do `olist.parquet` que viram features
2. Criar features derivadas (ex: diferença entre prazo previsto e real)
3. Definir o target binário (`review_positivo`)
4. Salvar como `features.parquet`

In [ ]:
import duckdb

FEATURES_PATH = "../data/features_olist_explorado.parquet"

duckdb.sql(f"""
COPY (
    SELECT
        order_id,

        -- Numéricas brutas
        price,
        freight_value,
        product_weight_g,
        product_length_cm,
        product_height_cm,
        product_width_cm,
        product_photos_qty,
        payment_installments,
        payment_value,
        delivery_days,
        estimated_delivery_days,

        -- Features derivadas
        (delivery_days - estimated_delivery_days)  AS delivery_diff,
        freight_value / NULLIF(price, 0)            AS freight_ratio,
        CASE WHEN delivered_late THEN 1 ELSE 0 END  AS delivered_late_int,

        -- Categóricas
        customer_state,
        seller_state,
        payment_type,
        product_category_en,
        (customer_state = seller_state) AS same_state,

        -- Target
        CASE WHEN review_positivo THEN 1 ELSE 0 END AS y_review_positivo
    FROM '../data/olist.parquet'
    WHERE review_positivo IS NOT NULL
      AND delivery_days IS NOT NULL
)
TO '{FEATURES_PATH}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")

df_features = duckdb.sql(f"SELECT * FROM '{FEATURES_PATH}'").df()
print(f"Linhas: {len(df_features):,}")
print(f"Colunas: {df_features.shape[1]}")
df_features.head()

### Olhando a distribuição do target

Quantos % dos reviews são positivos?

In [ ]:
import pandas as pd

prevalencia = df_features["y_review_positivo"].mean()
print(f"Prevalência (positivos): {prevalencia*100:.1f}%")
print(f"Negativos:                {(1-prevalencia)*100:.1f}%")

# Baseline ingênuo: chutar sempre "positivo" acertaria essa % das vezes
print(f"\nBaseline naive (sempre positivo): {prevalencia*100:.1f}% accuracy")
print("→ Nosso modelo precisa BATER esse baseline para ser útil.")

### Conhecendo as features numéricas

In [ ]:
numericas = [
    "price", "freight_value", "product_weight_g",
    "delivery_days", "estimated_delivery_days",
    "delivery_diff", "freight_ratio", "delivered_late_int",
]
df_features[numericas].describe().round(2)

### Insight: pedidos atrasados vs no prazo

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Barra: % de positivos por status de entrega
sumario = df_features.groupby("delivered_late_int").agg(
    pedidos=("y_review_positivo", "size"),
    pct_positivo=("y_review_positivo", "mean"),
).round(3)
sumario["pct_positivo"] *= 100
print(sumario)

(sumario["pct_positivo"] * 1).plot(
    kind="bar", ax=axes[0], color=["#2f9e44", "#e03131"]
)
axes[0].set_title("% de reviews positivos por status de entrega")
axes[0].set_xlabel("Atrasado?")
axes[0].set_ylabel("% positivos")
axes[0].set_xticklabels(["No prazo", "Atrasado"], rotation=0)

# Histograma: delivery_days
df_features["delivery_days"].clip(upper=60).hist(
    bins=40, ax=axes[1], color="#1971c2", edgecolor="white"
)
axes[1].set_title("Distribuição de dias de entrega")
axes[1].set_xlabel("Dias até a entrega (cap em 60)")
axes[1].set_ylabel("Pedidos")

plt.tight_layout()
plt.show()

**Observação:** pedidos no prazo têm taxa muito maior de reviews positivos. Atraso é o melhor preditor isolado — é por isso que vamos incluir essa feature no modelo.

### Mexa você mesmo

1. Quantas categorias diferentes de produto temos? Use `df_features["product_category_en"].nunique()`.
2. Existe diferença na prevalência de positivos por `payment_type`? Quem paga com cartão é mais ou menos satisfeito?

In [ ]:
# Sua exploração aqui!


---

## Etapa 2 — Treino do modelo com sklearn

> Equivalente: [`02_train.py`](./02_train.py)

Vamos construir um `Pipeline` do scikit-learn com:
- Pré-processamento (imputação + scaling + one-hot)
- Modelo (`RandomForestClassifier`)

**Por que Pipeline?** Porque o modelo + transformações viram um único objeto. Você salva, carrega e usa sem precisar reaplicar pré-processamento manualmente.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

NUMERIC = [
    "price", "freight_value", "product_weight_g",
    "product_length_cm", "product_height_cm", "product_width_cm",
    "product_photos_qty", "payment_installments", "payment_value",
    "delivery_days", "estimated_delivery_days",
    "delivery_diff", "freight_ratio", "delivered_late_int",
]
CATEG = [
    "customer_state", "seller_state", "payment_type",
    "product_category_en", "same_state",
]
TARGET = "y_review_positivo"

X = df_features[NUMERIC + CATEG]
y = df_features[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Treino: {len(X_train):,} linhas")
print(f"Teste:  {len(X_test):,} linhas")

In [ ]:
pre = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("sc", StandardScaler()),
        ]), NUMERIC),
        ("cat", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("oh", OneHotEncoder(handle_unknown="ignore", min_frequency=20)),
        ]), CATEG),
    ]
)
pipe = Pipeline([
    ("prep", pre),
    ("clf", RandomForestClassifier(
        n_estimators=200, max_depth=12, n_jobs=-1, random_state=42
    )),
])

import time
t0 = time.time()
pipe.fit(X_train, y_train)
print(f"Treino completo em {time.time()-t0:.1f}s")

### Avaliando o modelo no holdout

In [ ]:
y_pred = pipe.predict(X_test)
y_proba = pipe.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)

print(f"Accuracy: {acc:.3f}  (baseline naive: {y.mean():.3f})")
print(f"AUC:      {auc:.3f}")
print()
print(classification_report(y_test, y_pred, target_names=["NEGATIVO", "POSITIVO"]))

**Como ler:**
- **Precision** de POSITIVO ~ 0.82: quando o modelo diz "positivo", acerta 82% das vezes.
- **Recall** de NEGATIVO ~ 0.21: o modelo só pega 21% dos reviews realmente negativos (o problema clássico de imbalance).
- **AUC** ~ 0.69: discriminação acima do aleatório (0.5), mas longe do perfeito (1.0).

Modelo é **bom em confirmar positivos, ruim em pegar negativos**. Faz sentido — só 21% dos reviews são negativos, então o modelo tem pouco material para aprender.

### Matriz de confusão

In [ ]:
import numpy as np

cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(cm, cmap="Blues")
for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{cm[i,j]:,}",
                ha="center", va="center",
                color="white" if cm[i,j] > cm.max()/2 else "black",
                fontsize=14)
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(["NEG previsto", "POS previsto"])
ax.set_yticklabels(["NEG real", "POS real"])
ax.set_title("Matriz de confusão")
plt.tight_layout()
plt.show()

### Quais features mais importam?

O `RandomForest` consegue te dizer **quanto cada feature contribuiu** para as decisões.

In [ ]:
# Extrai os nomes das features depois do pré-processamento
nomes = pipe.named_steps["prep"].get_feature_names_out()
importancias = pipe.named_steps["clf"].feature_importances_

top = pd.DataFrame({
    "feature": nomes,
    "importance": importancias,
}).sort_values("importance", ascending=False).head(15)
print(top.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(top["feature"][::-1], top["importance"][::-1], color="#2f9e44")
ax.set_title("Top 15 features mais importantes")
ax.set_xlabel("Importância (RandomForest)")
plt.tight_layout()
plt.show()

**Quem domina?** Quase sempre as features de **atraso de entrega** e **tempo total** — confirmando o insight da Etapa 1.

### Salvando o modelo

In [ ]:
import joblib

MODEL_PATH = "../data/model_review_positivo_explorado.joblib"
joblib.dump(pipe, MODEL_PATH)
print(f"Modelo salvo em {MODEL_PATH} ({pathlib.Path(MODEL_PATH).stat().st_size/1024:.1f} KB)")

### Mexa você mesmo

1. Substitua `RandomForestClassifier` por `LogisticRegression()`. A acurácia sobe ou cai?
2. Aumente `n_estimators=500`. Quanto tempo demora a mais? Vale a pena?
3. Tente `class_weight="balanced"` no RandomForest. O recall de NEGATIVO melhora?

In [ ]:
# Sua versão!


---

## Etapa 3 — Predição em batch

> Equivalente: [`03_batch_predict.py`](./03_batch_predict.py)

Em produção, o modelo recebe lotes de novos pedidos e produz predições. Vamos simular isso aqui.

In [ ]:
# Carrega features (simulando "novos pedidos")
batch = duckdb.sql(f"""
    SELECT order_id, {", ".join(NUMERIC + CATEG)}
    FROM '{FEATURES_PATH}'
""").df()
print(f"Batch: {len(batch):,} pedidos")

# Carrega modelo
modelo = joblib.load(MODEL_PATH)

# Prediz
t0 = time.time()
proba = modelo.predict_proba(batch[NUMERIC + CATEG])[:, 1]
predicao = modelo.predict(batch[NUMERIC + CATEG])
print(f"Predição em {(time.time()-t0)*1000:.0f} ms")

# Constrói resultado
predicoes = pd.DataFrame({
    "order_id": batch["order_id"],
    "probabilidade_positivo": proba,
    "predicao": predicao,
})

# Salva em Parquet
PRED_PATH = "../data/predictions_olist_explorado.parquet"
predicoes.to_parquet(PRED_PATH, index=False)
print(f"Salvo: {PRED_PATH}")
predicoes.head()

### Distribuição das probabilidades

Onde o modelo está confiante e onde está em cima do muro?

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(predicoes["probabilidade_positivo"], bins=50,
        color="#1971c2", edgecolor="white")
ax.axvline(0.5, color="red", linestyle="--", label="Limiar 0.5")
ax.set_title("Distribuição das probabilidades preditas")
ax.set_xlabel("P(positivo)")
ax.set_ylabel("Pedidos")
ax.legend()
plt.tight_layout()
plt.show()

zona_duvida = predicoes[
    (predicoes["probabilidade_positivo"] > 0.4)
    & (predicoes["probabilidade_positivo"] < 0.6)
]
print(f"\nZona de dúvida (0.4 - 0.6): {len(zona_duvida):,} pedidos "
      f"({len(zona_duvida)/len(predicoes)*100:.1f}%)")

**Quando há muita massa em torno de 0.5**, o modelo está "em dúvida" — é exatamente o tipo de pedido onde **um humano (ou um LLM) poderia ajudar**. Esse é o gancho para a Aula 4.

---

## Etapa 4 — Avaliação por UF

> Equivalente: [`04_evaluate.py`](./04_evaluate.py)

Métricas globais escondem padrões. Vamos cortar a acurácia por estado para ver onde o modelo erra mais.

In [ ]:
avaliacao = duckdb.sql(f"""
SELECT
    o.customer_state AS uf,
    COUNT(*)                                                            AS pedidos,
    ROUND(AVG(p.probabilidade_positivo) * 100, 1)                       AS prob_media,
    ROUND(AVG(CASE WHEN o.review_positivo THEN 1 ELSE 0 END) * 100, 1)  AS pct_real_positivo,
    ROUND(
        AVG(
            CASE
                WHEN p.predicao = CASE WHEN o.review_positivo THEN 1 ELSE 0 END
                THEN 1 ELSE 0
            END
        ) * 100, 1
    )                                                                   AS acuracia
FROM '../data/olist.parquet' o
INNER JOIN '{PRED_PATH}' p USING (order_id)
GROUP BY uf
HAVING pedidos >= 100
ORDER BY pedidos DESC
LIMIT 12
""").df()

print(avaliacao.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(avaliacao["uf"], avaliacao["acuracia"], color="#2f9e44")
ax.axhline(y_pred.mean()*100 if False else 80, color="red", linestyle="--",
           label="Baseline (chutar sempre positivo)")
ax.set_title("Acurácia do modelo por UF")
ax.set_xlabel("UF")
ax.set_ylabel("Acurácia (%)")
ax.legend()
plt.tight_layout()
plt.show()

### Insight pedagógico

| Observação | Implicação |
|-----------|-----------|
| Acurácia é muito parecida entre estados | O modelo não está aprendendo viés regional forte |
| Estados com mais pedidos têm acurácia ligeiramente melhor | Mais dado de treino ajuda — confirma intuição |
| Toda UF está acima do baseline naive | O modelo está agregando valor, mesmo que modesto |

---

## Bloco bônus — Pra brincar

Algumas perguntas que dá para responder com este pipeline:

1. **A acurácia muda com o preço do pedido?** (pedidos caros vs baratos)
2. **A acurácia é melhor para reviews COM texto que para reviews SEM texto?**
3. **Qual categoria de produto é mais difícil de prever?**

### Mexa você mesmo: pergunta 1

In [ ]:
# Pedidos caros (top 25%) vs baratos (bottom 25%)
preco_p25 = df_features["price"].quantile(0.25)
preco_p75 = df_features["price"].quantile(0.75)
print(f"Bottom 25% (até R$ {preco_p25:.2f}) e top 25% (acima de R$ {preco_p75:.2f})")

# Junta predições com features
join = predicoes.merge(
    df_features[["order_id", "price", "y_review_positivo"]],
    on="order_id"
)
join["acertou"] = (join["predicao"] == join["y_review_positivo"]).astype(int)

# Acurácia por faixa de preço
join["faixa"] = pd.cut(
    join["price"],
    bins=[0, preco_p25, preco_p75, join["price"].max()],
    labels=["Barato (bottom 25%)", "Médio (25-75%)", "Caro (top 25%)"],
)

print()
print(join.groupby("faixa", observed=True)["acertou"]
      .agg(["mean", "size"])
      .rename(columns={"mean": "acuracia", "size": "pedidos"})
      .round(3))

## Próximos passos — Aula 4

Você acabou de construir um pipeline ML clássico em ~50 linhas de SQL e ~30 linhas de Python.

Na **Aula 4** vamos pegar exatamente esse mesmo problema (classificar review como positivo ou negativo) e atacar com um **agente de IA** que lê o `review_comment_message` em PT-BR. Sem features manuais. Sem treino. Só leitura de texto.

Pergunta: **quem ganha?**

| Abordagem | Treino | Features | Acurácia esperada |
|-----------|--------|----------|-------------------|
| RandomForest (este notebook) | ~2s | 19 numéricas + categóricas | ~82% |
| LLM (Aula 4) | 0s | só o texto do review | ? |

Te vejo lá. 🚀